# 🔤 Assamese OCR — Sentence Model Training (Kaggle)

This notebook trains a CRNN (CNN + LSTM) model for Assamese sentence-level OCR using CTC loss.

**Before you start:**
1. Set accelerator to **GPU** → Settings (right panel) → Accelerator → GPU T4 x2
2. Add your dataset containing `as-wiki-2021.txt` as a Kaggle Dataset input
3. (Optional) Add a dataset with your existing checkpoints `.pth` files
4. (Optional) Add a previously generated OCR dataset to skip regeneration

**Kaggle paths:**
- Input datasets (read-only): `/kaggle/input/<dataset-slug>/`
- Working directory (writable, **ephemeral**): `/kaggle/working/`

**⚠️ IMPORTANT:** `/kaggle/working/` is wiped when the runtime ends.  
This notebook auto-archives checkpoints & datasets as `.zip` files so they survive across sessions.

---

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL    = "https://github.com/dikshitadutta/AXOMIYA_OCR.git"
REPO_NAME   = "AXOMIYA_OCR"
BRANCH      = "main"
REPO_DIR    = f'/kaggle/working/{REPO_NAME}'
WORK_DIR    = f'{REPO_DIR}/django_backend'

# Clone ONLY if the repo doesn't already exist (avoids ephemeral re-cloning issues)
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    print(f'✅ Cloned {REPO_NAME} ({BRANCH})')
else:
    print(f'✅ Repo already exists at {REPO_DIR}')
    # Pull latest only if you explicitly want updates — otherwise skip
    # !cd {REPO_DIR} && git checkout {BRANCH} && git pull

%cd {WORK_DIR}
!pwd

In [ ]:
# Install dependencies
!pip install -q -r requirements-train.txt

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Enable GPU in Settings → Accelerator.')

## 2. Locate Input Data & Set Up Directories

Your `as-wiki-2021.txt` must be added as a Kaggle Dataset.

**How to add it:**
1. Upload `as-wiki-2021.txt` to a Kaggle Dataset (or use an existing one)
2. In this notebook → top-right → **+ Add Data** → search for your dataset
3. It will appear at `/kaggle/input/<your-dataset-slug>/as-wiki-2021.txt`

Update `WIKI_INPUT_PATH` below to match your dataset slug.

In [ ]:
import os
import shutil

# ─── UPDATE THESE to match your Kaggle dataset slugs ──────────────────────────
WIKI_INPUT_PATH     = '/kaggle/input/assamese-wiki/as-wiki-2021.txt'   # ← CHANGE
CHECKPOINT_INPUT    = '/kaggle/input/assamese-ocr-checkpoints'          # ← CHANGE
DATASET_INPUT       = '/kaggle/input/assamese-ocr-dataset'             # ← CHANGE (optional)
# ──────────────────────────────────────────────────────────────────────────────

# Verify the wiki file exists
if os.path.exists(WIKI_INPUT_PATH):
    size_mb = os.path.getsize(WIKI_INPUT_PATH) / 1e6
    print(f'✅ Wiki file found: {WIKI_INPUT_PATH} ({size_mb:.1f} MB)')
else:
    print(f'❌ Wiki file not found at: {WIKI_INPUT_PATH}')
    print('   Add your dataset via "+ Add Data" in the right panel.')

# Create local data/ and checkpoints/ directories
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Symlink wiki file into data/ for convenience
wiki_link = 'data/as-wiki-2021.txt'
if not os.path.exists(wiki_link):
    os.symlink(WIKI_INPUT_PATH, wiki_link)
    print(f'✅ Symlinked wiki file → data/as-wiki-2021.txt')
else:
    print(f'✅ data/as-wiki-2021.txt already linked')

In [ ]:
# Restore checkpoints from a previously saved Kaggle dataset
if os.path.exists(CHECKPOINT_INPUT):
    restored = 0
    for f in os.listdir(CHECKPOINT_INPUT):
        if f.endswith('.pth'):
            src = os.path.join(CHECKPOINT_INPUT, f)
            dst = os.path.join('checkpoints', f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f'  Copied: {f}')
                restored += 1
            else:
                print(f'  Already exists: {f}')
    print(f'✅ Checkpoints ready ({restored} restored)')
else:
    print('ℹ️  No checkpoint input dataset found — training from scratch.')

print('\nCheckpoints directory:')
!ls -lh checkpoints/ 2>/dev/null || echo '  (empty)'

## 3. Restore or Generate Train / Val / Test Splits

**Strategy:**
1. If a pre-generated dataset exists as a Kaggle Dataset input → **restore it** (fastest)
2. If splits already exist in working dir → **skip regeneration**
3. Only generate from scratch if nothing exists

This prevents the #1 problem: accidentally deleting and regenerating data every run.

In [ ]:
import os, shutil

train_labels = 'data/train_real_sentences/labels/labels.txt'
SPLITS = ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']

# ── Option A: Restore from a previously saved Kaggle dataset ──────────────────
if not os.path.exists(train_labels) and os.path.exists(DATASET_INPUT):
    print('📦 Restoring dataset from Kaggle input...')
    for split in SPLITS:
        src = os.path.join(DATASET_INPUT, split)
        dst = os.path.join('data', split)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print(f'  Restored: {split}')
    print('✅ Dataset restored from Kaggle input')

# ── Check current state ───────────────────────────────────────────────────────
if os.path.exists(train_labels):
    with open(train_labels) as f:
        n = sum(1 for _ in f)
    print(f'✅ Splits already exist! ({n} training samples)')
    print('⏩ Skipping generation — jump to Section 4 (Training).')
    NEED_GENERATION = False
else:
    print('❌ No splits found. Run the next cell to generate them.')
    NEED_GENERATION = True

In [ ]:
# ── Option B: Generate fresh splits (ONLY if needed) ─────────────────────────
# The build script splits TEXT FIRST, then renders images per split,
# so there is zero train/val text leakage.

if NEED_GENERATION:
    # Create clean output directories
    for split in SPLITS:
        dst = f'data/{split}'
        if os.path.islink(dst):
            os.unlink(dst)
        elif os.path.exists(dst):
            shutil.rmtree(dst)
        os.makedirs(f'{dst}/images', exist_ok=True)
        os.makedirs(f'{dst}/labels', exist_ok=True)

    !python build_real_sentence_splits.py \
        --input data/as-wiki-2021.txt \
        --train-output data/train_real_sentences \
        --val-output data/val_real_sentences \
        --test-output data/test_real_sentences \
        --train-count 50000 \
        --val-count 5000 \
        --test-count 2000 \
        --seed 42

    # Verify
    print()
    for split in SPLITS:
        img_dir    = f'data/{split}/images'
        label_file = f'data/{split}/labels/labels.txt'
        n_images   = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
        ok         = '✅' if os.path.exists(label_file) else '❌'
        print(f'{split}: {n_images} images, labels {ok}')
    print('\n✅ Dataset generation complete')
else:
    print('⏩ Dataset already exists — skipping generation.')

In [ ]:
# ── Archive generated dataset as a zip (for Kaggle Dataset upload) ────────────
# After generation, save a zip so you can create a permanent Kaggle Dataset.
# Go to Output tab → "New Dataset" to persist it across sessions.

import shutil

DATASET_ZIP = '/kaggle/working/assamese_ocr_dataset'
if not os.path.exists(f'{DATASET_ZIP}.zip'):
    print('📦 Archiving dataset to zip...')
    shutil.make_archive(DATASET_ZIP, 'zip', 'data')
    zip_size = os.path.getsize(f'{DATASET_ZIP}.zip') / 1e6
    print(f'✅ Saved: {DATASET_ZIP}.zip ({zip_size:.1f} MB)')
    print('\n💡 TIP: After the run, go to Output tab → "New Dataset" to save permanently.')
else:
    print(f'✅ Archive already exists: {DATASET_ZIP}.zip')

## 4. Train the Sentence Model 🚀

**Key features:**
- `--resume` restores optimizer, scheduler, epoch, and best metrics from the latest checkpoint
- `--latest-checkpoint` is saved **every epoch** (crash-safe)
- `--best-checkpoint` is saved only when validation loss improves
- Training auto-archives checkpoints to a zip after completion

In [ ]:
# Check if we should resume from an existing checkpoint
import os
LATEST_CKPT = 'checkpoints/latest_model_sentences.pth'
RESUME_FLAG = '--resume' if os.path.exists(LATEST_CKPT) else ''

if RESUME_FLAG:
    print(f'🔄 Resuming from: {LATEST_CKPT}')
else:
    print('🆕 Starting training from scratch')

In [ ]:
# Main training run — with automatic resume support
!python train_sentence_model.py \
    --train-img-dir   data/train_real_sentences/images \
    --train-label-file data/train_real_sentences/labels/labels.txt \
    --val-img-dir     data/val_real_sentences/images \
    --val-label-file  data/val_real_sentences/labels/labels.txt \
    --epochs 25 \
    --batch-size 32 \
    --learning-rate 0.0001 \
    --best-checkpoint  checkpoints/best_model_sentences.pth \
    --final-checkpoint checkpoints/final_model_sentences.pth \
    --latest-checkpoint checkpoints/latest_model_sentences.pth \
    --plot-out training_curve_sentences.png \
    {RESUME_FLAG}

In [ ]:
# View training curve
from IPython.display import Image as IPImage, display
if os.path.exists('training_curve_sentences.png'):
    display(IPImage('training_curve_sentences.png'))
else:
    print('No training curve found yet.')

## 5. Auto-Archive Checkpoints

**Critical:** Kaggle wipes `/kaggle/working/` when the session ends.  
This cell zips all checkpoints so they appear in the Output tab for download or dataset creation.

In [ ]:
import shutil, os

CKPT_ZIP = '/kaggle/working/checkpoints_backup'

if os.path.exists('checkpoints') and os.listdir('checkpoints'):
    shutil.make_archive(CKPT_ZIP, 'zip', 'checkpoints')
    zip_size = os.path.getsize(f'{CKPT_ZIP}.zip') / 1e6
    print(f'✅ Checkpoints archived: {CKPT_ZIP}.zip ({zip_size:.1f} MB)')
    print('\n💡 Go to Output tab → "New Dataset" to save as a Kaggle Dataset.')
    print('   Then add it as input next session to auto-restore checkpoints.')
else:
    print('⚠️  No checkpoints to archive.')

## 6. Evaluate Model Accuracy (CER / WER / Exact Match)

In [ ]:
# Evaluate on VALIDATION set — raw model output (greedy decode)
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --num-samples 10

In [ ]:
# Evaluate on VALIDATION set — with spell-check post-processing
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --post-process \
    --num-samples 10

In [ ]:
# Evaluate on VALIDATION set — CTC Beam Search
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --beam-width 10 \
    --num-samples 10

In [ ]:
# Evaluate on TEST set
test_labels = 'data/test_real_sentences/labels/labels.txt'
if os.path.exists(test_labels):
    !python evaluate.py \
        --img-dir    data/test_real_sentences/images \
        --label-file {test_labels} \
        --checkpoint checkpoints/best_model_sentences.pth \
        --num-samples 10 \
        --output test_results.tsv
    print('\n✅ Detailed results saved to test_results.tsv')
else:
    print('❌ No test set found. Regenerate splits with --test-count 2000.')

In [ ]:
# Quick single-image prediction
!python predict_cli.py \
    --image        data/val_real_sentences/images/sentence_000000.png \
    --checkpoint   checkpoints/best_model_sentences.pth \
    --ground-truth data/val_real_sentences/labels/sentence_000000.txt

## 7. (Optional) Transfer Learning

If you have `best_model_fast.pth` in `checkpoints/`, you can fine-tune from the character model.

In [ ]:
# Transfer learning — uncomment to run
# !python train_transfer_learning.py

## 8. Final Output Summary

**⚠️ REMINDER:** To persist files across sessions, you MUST either:
1. **Save Notebook Version** (File → Save Version) — saves outputs
2. **Create a Kaggle Dataset** from the Output tab — best for reuse

In [ ]:
import os

print('═' * 60)
print('📋 OUTPUT SUMMARY')
print('═' * 60)

print('\n=== Checkpoints ===')
!ls -lh checkpoints/ 2>/dev/null || echo '  (none)'

print('\n=== Archives (for Kaggle Dataset upload) ===')
!ls -lh /kaggle/working/*.zip 2>/dev/null || echo '  (none)'

print('\n=== Plots & Results ===')
!ls -lh *.png *.tsv 2>/dev/null || echo '  (none)'

print()
print('═' * 60)
print('⚠️  Files in /kaggle/working/ are EPHEMERAL.')
print('   → Save Notebook Version (File → Save Version)')
print('   → Or create Kaggle Datasets from the Output tab')
print('═' * 60)